In [1]:
# =========================================================
# TASK 19 — CELL 1
# ENVIRONMENT & CONFIGURATION
# =========================================================

import sys
import os
import json
import joblib
import numpy as np
import pandas as pd
import sklearn

from pathlib import Path

print("=" * 65)
print("       TASK 19 — APPLICATION MODEL SERIALIZATION")
print("=" * 65)

RANDOM_STATE = 42
MODEL_VERSION = "v1"

PROJECT_ROOT = Path("/home/akash/Projects/Altrodav")

MODELS_DIR = PROJECT_ROOT / "models"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
SRC_DIR = PROJECT_ROOT / "src"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODELS_DIR / f"task19_iris_pipeline_{MODEL_VERSION}.joblib"
METADATA_PATH = ARTIFACTS_DIR / f"task19_model_metadata_{MODEL_VERSION}.json"
PREDICTIONS_PATH = ARTIFACTS_DIR / f"task19_test_predictions_{MODEL_VERSION}.csv"
VERIFICATION_PATH = ARTIFACTS_DIR / f"task19_serialization_verification_{MODEL_VERSION}.json"

print("\n========== ENVIRONMENT ==========")
print("Python version :", sys.version.split()[0])
print("NumPy version  :", np.__version__)
print("Pandas version :", pd.__version__)
print("Scikit-learn  :", sklearn.__version__)
print("Joblib version :", joblib.__version__)

print("\n========== CONFIGURATION ==========")
print("Random state   :", RANDOM_STATE)
print("Model version  :", MODEL_VERSION)

print("\n========== PATHS ==========")
print("Project root   :", PROJECT_ROOT)
print("Model path     :", MODEL_PATH)
print("Metadata path  :", METADATA_PATH)

print("\nEnvironment and configuration setup completed successfully.")

       TASK 19 — APPLICATION MODEL SERIALIZATION

========== ENVIRONMENT ==========
Python version : 3.13.12
NumPy version  : 2.5.1
Pandas version : 2.3.3
Scikit-learn  : 1.9.0
Joblib version : 1.5.3

========== CONFIGURATION ==========
Random state   : 42
Model version  : v1

========== PATHS ==========
Project root   : /home/akash/Projects/Altrodav
Model path     : /home/akash/Projects/Altrodav/models/task19_iris_pipeline_v1.joblib
Metadata path  : /home/akash/Projects/Altrodav/artifacts/task19_model_metadata_v1.json

Environment and configuration setup completed successfully.


In [2]:
# =========================================================
# TASK 19 — CELL 2
# LOAD DATASET
# =========================================================

from sklearn.datasets import load_iris

iris = load_iris()

X = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

y = pd.Series(
    iris.target,
    name="target"
)

class_names = list(iris.target_names)

print("========== DATASET ==========")

print("Dataset shape:", X.shape)

print("\nFeature columns:")
print(list(X.columns))

print("\nTarget distribution:")
print(y.value_counts().sort_index())

print("\nTarget classes:")
for i, name in enumerate(class_names):
    print(f"{i}: {name}")

print("\nDataset loaded successfully.")

========== DATASET ==========
Dataset shape: (150, 4)

Feature columns:
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

Target distribution:
target
0    50
1    50
2    50
Name: count, dtype: int64

Target classes:
0: setosa
1: versicolor
2: virginica

Dataset loaded successfully.


In [3]:
# =========================================================
# TASK 19 — CELL 3
# TRAIN / TEST SPLIT
# =========================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("========== TRAIN / TEST SPLIT ==========")

print("Training samples:", len(X_train))
print("Test samples    :", len(X_test))

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())

print("\nTest target distribution:")
print(y_test.value_counts().sort_index())

print("\nTest set remains completely independent.")

========== TRAIN / TEST SPLIT ==========
Training samples: 120
Test samples    : 30

Training target distribution:
target
0    40
1    40
2    40
Name: count, dtype: int64

Test target distribution:
target
0    10
1    10
2    10
Name: count, dtype: int64

Test set remains completely independent.


In [4]:
# =========================================================
# TASK 19 — CELL 4
# BUILD PREPROCESSING + MODEL PIPELINE
# =========================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "model",
            SVC(
                C=1.0,
                kernel="rbf",
                gamma="scale",
                probability=True,
                random_state=RANDOM_STATE
            )
        )
    ]
)

print("========== PIPELINE ==========")

print(pipeline)

print("\nPipeline steps:")
for name, step in pipeline.named_steps.items():
    print(f"- {name}: {type(step).__name__}")

print("\nComplete preprocessing + model pipeline created successfully.")

========== PIPELINE ==========
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', SVC(probability=True, random_state=42))])

Pipeline steps:
- scaler: StandardScaler
- model: SVC

Complete preprocessing + model pipeline created successfully.


In [5]:
# =========================================================
# TASK 19 — CELL 5
# TRAIN PIPELINE
# =========================================================

pipeline.fit(X_train, y_train)

train_predictions = pipeline.predict(X_train)
test_predictions = pipeline.predict(X_test)

train_accuracy = np.mean(train_predictions == y_train)
test_accuracy = np.mean(test_predictions == y_test)

print("========== MODEL TRAINING ==========")

print("Training completed successfully.")

print("\nTraining accuracy:", round(train_accuracy, 4))
print("Test accuracy    :", round(test_accuracy, 4))

print("\nPipeline training completed successfully.")

========== MODEL TRAINING ==========
Training completed successfully.

Training accuracy: 0.975
Test accuracy    : 0.9667

Pipeline training completed successfully.


/home/akash/Projects/Altrodav/venv/lib/python3.13/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [6]:
# =========================================================
# TASK 19 — CELL 6
# MODEL EVALUATION
# =========================================================

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

test_accuracy = accuracy_score(
    y_test,
    test_predictions
)

print("========== MODEL EVALUATION ==========")

print("Training accuracy:", round(train_accuracy, 4))
print("Test accuracy    :", round(test_accuracy, 4))

print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_test,
        test_predictions,
        target_names=class_names
    )
)

print("Model evaluation completed successfully.")

========== MODEL EVALUATION ==========
Training accuracy: 0.975
Test accuracy    : 0.9667

========== CLASSIFICATION REPORT ==========
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

Model evaluation completed successfully.


In [7]:
# =========================================================
# TASK 19 — CELL 7
# SAVE TEST PREDICTIONS
# =========================================================

test_results = X_test.copy()

test_results["actual_class"] = y_test.values
test_results["predicted_class"] = test_predictions

test_results["actual_name"] = [
    class_names[int(value)]
    for value in y_test
]

test_results["predicted_name"] = [
    class_names[int(value)]
    for value in test_predictions
]

test_results.to_csv(
    PREDICTIONS_PATH,
    index=False
)

print("========== TEST PREDICTIONS ==========")

print("Prediction artifact saved:")
print(PREDICTIONS_PATH)

print("\nPrediction samples:")
print(test_results.head())

print("\nTest prediction artifact created successfully.")

========== TEST PREDICTIONS ==========
Prediction artifact saved:
/home/akash/Projects/Altrodav/artifacts/task19_test_predictions_v1.csv

Prediction samples:
     sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
38                 4.4               3.0                1.3               0.2   
127                6.1               3.0                4.9               1.8   
57                 4.9               2.4                3.3               1.0   
93                 5.0               2.3                3.3               1.0   
42                 4.4               3.2                1.3               0.2   

     actual_class  predicted_class actual_name predicted_name  
38              0                0      setosa         setosa  
127             2                2   virginica      virginica  
57              1                1  versicolor     versicolor  
93              1                1  versicolor     versicolor  
42              0                0 

In [8]:
# =========================================================
# TASK 19 — CELL 8
# CREATE MODEL METADATA
# =========================================================

metadata = {
    "task": "Task 19 — Application Model Serialization",
    "model_name": "Iris SVM Classification Pipeline",
    "model_version": MODEL_VERSION,

    "dataset": {
        "name": "Iris Dataset",
        "source": "sklearn.datasets.load_iris",
        "total_samples": int(len(X)),
        "training_samples": int(len(X_train)),
        "test_samples": int(len(X_test)),
        "number_of_features": int(X.shape[1]),
        "number_of_classes": int(len(class_names))
    },

    "features": list(X.columns),

    "classes": {
        str(i): name
        for i, name in enumerate(class_names)
    },

    "preprocessing": {
        "method": "StandardScaler"
    },

    "model": {
        "type": "SVC",
        "C": 1.0,
        "kernel": "rbf",
        "gamma": "scale",
        "probability": True,
        "random_state": RANDOM_STATE
    },

    "metrics": {
        "training_accuracy": round(float(train_accuracy), 4),
        "test_accuracy": round(float(test_accuracy), 4)
    },

    "serialization": {
        "format": "joblib",
        "artifact": MODEL_PATH.name
    },

    "environment": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": joblib.__version__
    }
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=4)

print("========== MODEL METADATA ==========")

print("Metadata saved successfully:")
print(METADATA_PATH)

print("\nModel metadata:")
print(json.dumps(metadata, indent=4))

========== MODEL METADATA ==========
Metadata saved successfully:
/home/akash/Projects/Altrodav/artifacts/task19_model_metadata_v1.json

Model metadata:
{
    "task": "Task 19 \u2014 Application Model Serialization",
    "model_name": "Iris SVM Classification Pipeline",
    "model_version": "v1",
    "dataset": {
        "name": "Iris Dataset",
        "source": "sklearn.datasets.load_iris",
        "total_samples": 150,
        "training_samples": 120,
        "test_samples": 30,
        "number_of_features": 4,
        "number_of_classes": 3
    },
    "features": [
        "sepal length (cm)",
        "sepal width (cm)",
        "petal length (cm)",
        "petal width (cm)"
    ],
    "classes": {
        "0": "setosa",
        "1": "versicolor",
        "2": "virginica"
    },
    "preprocessing": {
        "method": "StandardScaler"
    },
    "model": {
        "type": "SVC",
        "C": 1.0,
        "kernel": "rbf",
        "gamma": "scale",
        "probability": true,
     

In [9]:
# =========================================================
# TASK 19 — CELL 9
# SERIALIZE COMPLETE PIPELINE
# =========================================================

joblib.dump(
    pipeline,
    MODEL_PATH
)

print("========== MODEL SERIALIZATION ==========")

print("Serialized artifact:")
print(MODEL_PATH)

print("\nFile exists:", MODEL_PATH.exists())

if MODEL_PATH.exists():
    print("File size:", round(MODEL_PATH.stat().st_size / 1024, 2), "KB")

print("\nComplete preprocessing + model pipeline serialized successfully.")

========== MODEL SERIALIZATION ==========
Serialized artifact:
/home/akash/Projects/Altrodav/models/task19_iris_pipeline_v1.joblib

File exists: True
File size: 5.69 KB

Complete preprocessing + model pipeline serialized successfully.


In [10]:
# =========================================================
# TASK 19 — CELL 10
# LOAD SERIALIZED PIPELINE
# =========================================================

loaded_pipeline = joblib.load(MODEL_PATH)

print("========== PIPELINE LOADING ==========")

print("Loaded artifact:")
print(MODEL_PATH)

print("\nLoaded pipeline:")
print(loaded_pipeline)

print("\nPipeline type:")
print(type(loaded_pipeline).__name__)

print("\nPipeline loading completed successfully.")

========== PIPELINE LOADING ==========
Loaded artifact:
/home/akash/Projects/Altrodav/models/task19_iris_pipeline_v1.joblib

Loaded pipeline:
Pipeline(steps=[('scaler', StandardScaler()),
                ('model', SVC(probability=True, random_state=42))])

Pipeline type:
Pipeline

Pipeline loading completed successfully.


In [11]:
# =========================================================
# TASK 19 — CELL 11
# INPUT VALIDATION + LOAD & PREDICT FUNCTION
# =========================================================

FEATURE_NAMES = list(X.columns)

def validate_input(features):
    """
    Validate incoming Iris feature values.
    """

    if not isinstance(features, dict):
        raise TypeError("Input must be a dictionary.")

    missing_features = [
        feature
        for feature in FEATURE_NAMES
        if feature not in features
    ]

    if missing_features:
        raise ValueError(
            f"Missing features: {missing_features}"
        )

    validated_values = []

    for feature in FEATURE_NAMES:

        value = features[feature]

        if not isinstance(value, (int, float, np.integer, np.floating)):
            raise TypeError(
                f"{feature} must be numeric."
            )

        if not np.isfinite(value):
            raise ValueError(
                f"{feature} must be finite."
            )

        validated_values.append(float(value))

    return pd.DataFrame(
        [validated_values],
        columns=FEATURE_NAMES
    )


def load_and_predict(features):
    """
    Load serialized pipeline and generate prediction.
    """

    model = joblib.load(MODEL_PATH)

    validated_input = validate_input(features)

    prediction = int(
        model.predict(validated_input)[0]
    )

    result = {
        "predicted_class_id": prediction,
        "predicted_class": class_names[prediction]
    }

    if hasattr(model, "predict_proba"):

        probabilities = model.predict_proba(
            validated_input
        )[0]

        result["probabilities"] = {
            class_names[i]: round(float(probability), 4)
            for i, probability in enumerate(probabilities)
        }

    return result


print("========== PREDICTION FUNCTION ==========")

print("Input validation function created.")
print("Load-and-predict function created.")

print("\nPrediction function ready for application use.")

========== PREDICTION FUNCTION ==========
Input validation function created.
Load-and-predict function created.

Prediction function ready for application use.


In [12]:
# =========================================================
# TASK 19 — CELL 12
# TEST APPLICATION PREDICTION
# =========================================================

sample_input = {
    "sepal length (cm)": 5.1,
    "sepal width (cm)": 3.5,
    "petal length (cm)": 1.4,
    "petal width (cm)": 0.2
}

prediction_result = load_and_predict(
    sample_input
)

print("========== APPLICATION PREDICTION ==========")

print("Input:")
print(sample_input)

print("\nPrediction:")
print(prediction_result)

print("\nApplication prediction completed successfully.")

========== APPLICATION PREDICTION ==========
Input:
{'sepal length (cm)': 5.1, 'sepal width (cm)': 3.5, 'petal length (cm)': 1.4, 'petal width (cm)': 0.2}

Prediction:
{'predicted_class_id': 0, 'predicted_class': np.str_('setosa'), 'probabilities': {np.str_('setosa'): 0.9721, np.str_('versicolor'): 0.016, np.str_('virginica'): 0.0119}}

Application prediction completed successfully.
